# Classifying Stance in SemEval-2016 dataset using OpenAI model

In [35]:
from datasets import load_dataset
import json
from openai import OpenAI
import huggingface_hub
from IPython.display import Image, display
from dotenv import load_dotenv
import os
from sklearn.metrics import f1_score
import pandas as pd
from tabulate import tabulate

## API keys and logins

In [2]:
# Get api keys
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")

In [3]:
# Log in to huggingface to access private models and datasets
huggingface_hub.login(token=HUGGINGFACE_API_KEY)

# Initializing OpenAI client - see https://platform.openai.com/docs/quickstart?context=python
client = OpenAI(api_key=OPENAI_API_KEY)

## Data loading and formatting

In [5]:
ds = load_dataset("FundSciImpact/semeval-2016", split="test")

In [60]:
def format_example(example, idx):
    """Formats an example into OpenAI's Batch API structure"""
    return {"custom_id": f"row-{idx}",  #unique identifier
            "method": "POST", 
            "url": "/v1/chat/completions", 
            "body": {"model": "gpt-4o-mini", "temperature": 0.0, 
                     "messages": example}}

## Non-batch API testing

In [7]:
def get_stance(example):
    response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.0,
    # This is to enable JSON mode, making sure responses are valid json objects
    # response_format={"type": "json_object"},
    messages=example,
    )

    return response.choices[0].message.content

In [8]:
# We use messages_1 as the prompt ends with "Your answer should contain only a single word taken from the list [support, against, none]."
# We also remove the final {'content': '', 'role': 'assistant'} dictionary
for message in ds["messages_1"][:5]:
    result = get_stance(message[:-1])
    print(f"RESULT: {result}")
    print("\n\n----------------------------\n\n")

RESULT: none


----------------------------


RESULT: support


----------------------------


RESULT: against


----------------------------


RESULT: support


----------------------------


RESULT: support


----------------------------




## Batch API submission

In [9]:
tasks = [format_example(message[:-1], idx) for idx, message in enumerate(ds["messages_1"])]

In [10]:
# Creating the file

file_name = "batch_stance_semeval.jsonl"

with open(file_name, 'w') as file:
    for obj in tasks:
        file.write(json.dumps(obj) + '\n')

In [61]:
# Uploading the file

batch_file = client.files.create(file=open(file_name, "rb"), purpose="batch")

In [62]:
print(batch_file)

FileObject(id='file-Di88hhVr1ZLk3msWPZfnJv', bytes=2519431, created_at=1743597133, filename='batch_stance_semeval.jsonl', object='file', purpose='batch', status='processed', status_details=None, expires_at=None)


In [63]:
batch_job = client.batches.create(input_file_id=batch_file.id,
                                  endpoint="/v1/chat/completions",
                                  completion_window="24h"
                                 )

In [71]:
batch_job = client.batches.retrieve(batch_job.id)
print(batch_job)

Batch(id='batch_67ed2e52bfac819097bf1581f3658419', completion_window='24h', created_at=1743597138, endpoint='/v1/chat/completions', input_file_id='file-Di88hhVr1ZLk3msWPZfnJv', object='batch', status='in_progress', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1743683538, failed_at=None, finalizing_at=None, in_progress_at=1743597140, metadata=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=1956))


In [16]:
# When finished
result_file_id = batch_job.output_file_id
result = client.files.content(result_file_id).content

In [17]:
# Save the results into a JSON file

result_file_name = "batch_stance_semeval_results.jsonl"

with open(result_file_name, 'wb') as file:
    file.write(result)